In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
### core fundataion - llm
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

### Node 1: JD Parser

In [3]:
from pydantic import BaseModel, Field
from typing import List

class JDStructured(BaseModel):
    role: str = Field(description="Job role/title")
    required_skills: List[str] = Field(description="Must-have skills")
    experience_years: int = Field(description="Years of experience required")
    location: str = Field(description="Job location")

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert recruiter who extracts structured data from job descriptions."),
    ("user", """
Extract the following from the job description:

- role
- required_skills
- experience_years
- location

Return ONLY valid JSON.

Job Description:
{jd}
""")
])

In [6]:
structured_llm = llm.with_structured_output(JDStructured)

chain = prompt | structured_llm

In [7]:
jd_text = """
We are looking for a Data Scientist with 3+ years of experience.
Must have strong Python, Machine Learning, and SQL skills.
Experience with AWS is a plus.
Location: Bangalore
"""

jd_parsed = chain.invoke({"jd": jd_text})

print(jd_parsed)

role='Data Scientist' required_skills=['Python', 'Machine Learning', 'SQL'] experience_years=3 location='Bangalore'


In [8]:
jd_parsed

JDStructured(role='Data Scientist', required_skills=['Python', 'Machine Learning', 'SQL'], experience_years=3, location='Bangalore')

### Node 2: Candidate Retriever